# 07 — Fine-tune a seq2seq model (AraBART / AraT5 / mT5)

**One notebook, any model.** Set `MODEL_KEY` in Step 0 and everything else follows from
`MODEL_CONFIGS`. Every model therefore goes through byte-identical data loading, tokenisation,
collation, evaluation and scoring — which is exactly what the paper's controlled comparison
claims. Maintaining three near-copies invites silent drift, and this project has already been
bitten by that (an earlier AraT5 notebook trained on AraSum only, with a different batch size,
making the three-way comparison invalid).

**What differs per model, and why** — all documented in `MODEL_CONFIGS`, none of it accidental:

| | AraBART | AraT5v2-base | mT5-small |
|---|---|---|---|
| family | BART | T5 | T5 |
| `fp16` | **True** | False | False |
| optimiser | AdamW | Adafactor | Adafactor |
| learning rate | 5e-5 | 5e-4 | 5e-4 |

T5-family models produce NaN loss under fp16 — a documented, unfixed architectural issue (they
were pretrained in bfloat16 and overflow fp16's range). bf16 is not an alternative on a T4:
it needs compute capability ≥ 8.0 and a T4 is 7.5. BART is unaffected, so it keeps fp16 and
trains considerably faster. Learning rate follows each family's published recipe, and the T5
recipe assumes Adafactor rather than AdamW.

Everything else — effective batch size 16, 512/100 token limits, 8 epochs, beam 4, seed 42, the
canonical split — is held constant by design.

**Resume is built in.** Set `RESUME_FROM_CHECKPOINT` to continue an interrupted run; leave it
`None` for a fresh one. AraT5 at fp32 will likely need more than one 12-hour Kaggle session.

**`TEST_MODE = True`** validates the whole notebook in ~2 minutes without training.

## Step 0 — Setup

In [ ]:
!nvidia-smi


`nvidia-smi` queries the driver directly, so it is ground truth for how many GPUs this session
has and whether anything is already holding memory on GPU 0. If GPU 0 already shows several GiB
in use, restart the kernel before continuing.

In [ ]:
!pip install -q evaluate rouge_score bert_score absl-py nltk qalsadi


### Model selection

Change `MODEL_KEY` only. Everything downstream reads from `MODEL_CONFIGS`, so a model swap
cannot silently change the training or evaluation protocol.

In [ ]:
import os

print(f"CUDA_VISIBLE_DEVICES before override: {os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>')}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"CUDA_VISIBLE_DEVICES after override:  {os.environ['CUDA_VISIBLE_DEVICES']}")

import re, json, time, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import evaluate
import nltk
from nltk.stem import SnowballStemmer
from rouge_score import rouge_scorer
import qalsadi.lemmatizer
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq,
)
from transformers.optimization import Adafactor
from tqdm import tqdm

nltk.download("stopwords", quiet=True)
tqdm.pandas()
print(f"Visible GPUs: {torch.cuda.device_count()} (pinned to 1 by design)")

# =======================  CHANGE THIS LINE ONLY  =======================
MODEL_KEY = "arabart"          # "arabart" | "arat5" | "mt5-small"
# =======================================================================

MODEL_CONFIGS = {
    "arabart": {
        "model_name": "moussaKam/AraBART",
        "family": "BART",
        "learning_rate": 5e-5,      # BART paper's fine-tuning range (3e-5 .. 5e-5)
        "use_fp16": True,           # BART is not affected by the T5 fp16/NaN issue
        "use_adafactor": False,     # BART is conventionally trained with AdamW
        "train_batch_size": 4,      # fp16 -> lower memory, so a larger per-device batch fits
        "grad_accum_steps": 4,      # 4 x 4 = 16 effective, same as every other model
    },
    "arat5": {
        "model_name": "UBC-NLP/AraT5v2-base-1024",
        "family": "T5",
        "learning_rate": 5e-4,      # T5 recipe; pairs with Adafactor
        "use_fp16": False,          # T5-family -> NaN under fp16
        "use_adafactor": True,
        "train_batch_size": 2,      # fp32 + base-sized model -> smaller batch
        "grad_accum_steps": 8,      # 2 x 8 = 16 effective
    },
    "mt5-small": {
        "model_name": "google/mt5-small",
        "family": "T5",
        "learning_rate": 5e-4,
        "use_fp16": False,
        "use_adafactor": True,
        "train_batch_size": 2,
        "grad_accum_steps": 8,
    },
}
assert MODEL_KEY in MODEL_CONFIGS, f"unknown MODEL_KEY: {MODEL_KEY}"
CFG = MODEL_CONFIGS[MODEL_KEY]

MODEL_NAME        = CFG["model_name"]
LEARNING_RATE     = CFG["learning_rate"]
USE_FP16          = CFG["use_fp16"]
USE_ADAFACTOR     = CFG["use_adafactor"]
TRAIN_BATCH_SIZE  = CFG["train_batch_size"]
GRAD_ACCUM_STEPS  = CFG["grad_accum_steps"]
EVAL_BATCH_SIZE   = TRAIN_BATCH_SIZE

# ---- Held constant across every model: changing these breaks comparability ----
MAX_INPUT_LEN  = 512      # AraT5v2 supports 1024, deliberately capped to match the others
MAX_TARGET_LEN = 100
GEN_MAX_LEN    = 150
NUM_BEAMS      = 4
NUM_EPOCHS     = 8
RANDOM_SEED    = 42
BATCH_SIZE_GEN = 8

TEST_MODE   = False       # True -> skip training, evaluate a subset, ~2 min end-to-end check
TEST_MODE_N = 20

DATA_DIR = "/kaggle/input/datasets/mohamedbentalb/arabic-summ-canonical-v1/canonical_dataset"
RESUME_FROM_CHECKPOINT = None    # set to a checkpoint dir to continue an interrupted run
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

assert TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS == 16, (
    "effective batch size must be 16 for every model, or the comparison is confounded"
)

print(f"\n=== {MODEL_KEY} ({CFG['family']}-family) ===")
print(f"  {MODEL_NAME}")
print(f"  lr={LEARNING_RATE}  fp16={USE_FP16}  optimizer={'Adafactor' if USE_ADAFACTOR else 'AdamW'}")
print(f"  batch {TRAIN_BATCH_SIZE} x accum {GRAD_ACCUM_STEPS} = {TRAIN_BATCH_SIZE*GRAD_ACCUM_STEPS} effective")
print(f"  epochs={NUM_EPOCHS}  seed={RANDOM_SEED}  TEST_MODE={TEST_MODE}")

for label, path, marker in [("data", DATA_DIR, "manifest.json"),
                            ("checkpoint", RESUME_FROM_CHECKPOINT, "trainer_state.json")]:
    if path is None:
        continue
    if os.path.exists(os.path.join(path, marker)):
        print(f"{label}: OK")
    else:
        print(f"\nWARNING: no {marker} for {label} at {path}. Searching /kaggle ...")
        for base in ["/kaggle/input", "/kaggle/working"]:
            for dirpath, _, filenames in os.walk(base):
                if marker in filenames:
                    print(f"  Found: {dirpath}")

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

# ---- GPU / precision advisory -------------------------------------------------
# Kaggle offers T4 x2 (Turing, tensor cores, 320 GB/s) and P100 (Pascal, no tensor
# cores, 732 GB/s). Neither is universally better: T4 wins on fp16, P100 on fp32.
if torch.cuda.is_available():
    _gpu = torch.cuda.get_device_name(0)
    _cc = torch.cuda.get_device_capability(0)
    _has_tc = _cc[0] >= 7          # Volta and newer have tensor cores
    if USE_FP16 and not _has_tc:
        print(f"\nADVISORY: fp16 requested but {_gpu} (compute capability {_cc[0]}.{_cc[1]}) "
              "has no tensor cores, so fp16 buys little here. Switch the Kaggle accelerator "
              "to T4 for this model.")
    elif (not USE_FP16) and "T4" in _gpu:
        print(f"\nADVISORY: this run is fp32 (required for T5-family models). A P100 has "
              "~1.2x the FP32 throughput and ~2.3x the memory bandwidth of a T4, so it is "
              "usually faster for fp32 training. Consider switching the accelerator to P100. "
              "Checkpoints are portable between them, so a resumed run can change GPU.")
    elif USE_FP16 and _has_tc:
        print(f"\nGPU/precision pairing looks right: fp16 on {_gpu} (tensor cores available).")
    else:
        print(f"\nGPU/precision pairing looks right: fp32 on {_gpu}.")


## Step 1 — Load the canonical split

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

with open(f"{DATA_DIR}/manifest.json") as f:
    data_manifest = json.load(f)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"Canonical dataset built at: {data_manifest['created_at_utc']}")

assert set(train_df["article_hash"]).isdisjoint(set(test_df["article_hash"])), "train/test overlap!"
assert set(val_df["article_hash"]).isdisjoint(set(test_df["article_hash"])), "val/test overlap!"
print("Leakage re-check passed.")


## Step 2 — GPU memory check

If a previous cell crashed, its tensors stay referenced by the traceback and
`torch.cuda.empty_cache()` alone cannot reclaim them. This reports what is actually free.

In [ ]:
import gc, sys

for _v in ["model", "trainer", "optimizer"]:
    if _v in dir():
        exec(f"del {_v}")
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU 0: {alloc:.2f} GiB allocated of {total:.2f} GiB")
    print("Looks clean." if alloc < 1.0 else
          "WARNING: memory already in use -- restart the kernel before continuing.")


## Step 3 — Load model and tokenizer

`tie_weights()` is **not** called here. Some checkpoints store the embedding matrix redundantly;
tying it before the optimiser is constructed changes how many distinct parameter tensors the
optimiser sees, which breaks `resume_from_checkpoint` with a parameter-group mismatch. Step 8
ties weights after training, so the saved model is still deduplicated.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME} on {device}  ({n_params/1e6:.0f}M parameters)")


## Step 4 — Tokenize

Labels are **not** padded here. Padding is deferred to `DataCollatorForSeq2Seq`, which fills
padded label positions with `-100` so the loss ignores them. Padding labels to a fixed length
with the real pad-token id instead trains the model to predict "pad" everywhere and collapses
it to empty output.

In [ ]:
def tokenize_function(examples):
    model_inputs = tokenizer(examples["article"], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(examples["summary"], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = Dataset.from_pandas(train_df[["article", "summary"]]).map(tokenize_function, batched=True)
eval_dataset  = Dataset.from_pandas(val_df[["article", "summary"]]).map(tokenize_function, batched=True)
print(f"train_dataset: {len(train_dataset)}  eval_dataset: {len(eval_dataset)}")


## Step 5 — Pre-flight

For a resumed run this reports what the checkpoint already contains. For a fresh run it
confirms training starts at epoch 0.

In [ ]:
ckpt_state = None
epochs_done = 0.0

if RESUME_FROM_CHECKPOINT is not None:
    with open(f"{RESUME_FROM_CHECKPOINT}/trainer_state.json") as f:
        ckpt_state = json.load(f)
    epochs_done = ckpt_state.get("epoch", 0)
    prior = [h["eval_loss"] for h in ckpt_state.get("log_history", []) if "eval_loss" in h]
    print(f"Checkpoint: epoch {epochs_done:.2f}, step {ckpt_state.get('global_step', 0)}")
    for i, v in enumerate(prior, 1):
        print(f"  Epoch {i}: {v:.4f}")
    print(f"This run trains up to epoch {NUM_EPOCHS} (~{NUM_EPOCHS - epochs_done:.0f} more).")
    assert epochs_done < NUM_EPOCHS, "checkpoint already at NUM_EPOCHS -- use 05_evaluate_checkpoint.ipynb"
else:
    print(f"Fresh run: training {MODEL_KEY} from pretrained weights for {NUM_EPOCHS} epochs.")

steps_per_epoch = len(train_dataset) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
print(f"~{steps_per_epoch:,} optimiser steps/epoch, ~{steps_per_epoch*NUM_EPOCHS:,} total.")


## Step 6 — Train

Do **not** interrupt this cell. An interrupted `trainer.train()` leaves the in-memory model in
an inconsistent state and it will emit non-Arabic token spam; files on disk are unaffected. Use
`TEST_MODE` to exercise the downstream cells instead. Step 10 has a sanity gate that catches a
corrupted model in seconds rather than after a full generation pass.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir=f"{RESULTS_DIR}/checkpoints_{MODEL_KEY}",
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    fp16=USE_FP16,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=1,
    seed=RANDOM_SEED,
)

# Adafactor is constructed explicitly, not via optim="adafactor": its own defaults
# (relative_step=True) compute an internal LR and would silently ignore LEARNING_RATE.
if USE_ADAFACTOR:
    optimizer = Adafactor(model.parameters(), lr=LEARNING_RATE,
                          scale_parameter=False, relative_step=False, warmup_init=False)
    optimizers = (optimizer, None)
    print("Optimizer: Adafactor (explicit LR)")
else:
    optimizers = (None, None)   # Trainer builds its default AdamW
    print("Optimizer: AdamW (Trainer default)")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    optimizers=optimizers,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

torch.cuda.empty_cache()
train_result = None
TRAINING_COMPLETED = False

if TEST_MODE:
    print("TEST_MODE -- skipping training.")
    if RESUME_FROM_CHECKPOINT is not None:
        model = AutoModelForSeq2SeqLM.from_pretrained(RESUME_FROM_CHECKPOINT).to(device)
        trainer.model = model
        print("Loaded checkpoint weights for evaluation.")
    else:
        print("No checkpoint set -- evaluating the untrained pretrained model.")
else:
    if RESUME_FROM_CHECKPOINT is not None:
        print(f"Resuming from {RESUME_FROM_CHECKPOINT}")
        train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    else:
        print(f"Training {MODEL_KEY} from scratch ({NUM_EPOCHS} epochs)")
        train_result = trainer.train()
    TRAINING_COMPLETED = True
    print(train_result)


## Step 7 — Training curve

Both series plot against the real `epoch` field in `log_history`, not a positional index — on a
resumed run the earlier and later epochs may have been logged at different frequencies, and a
positional axis would distort the curve.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
if TEST_MODE and not history and ckpt_state is not None:
    # TEST_MODE never ran training, so pull the curve straight from the checkpoint's own state.
    history = ckpt_state.get("log_history", [])
    print("TEST_MODE: plotting the checkpoint's stored history (epochs 1-5).")
train_pts = [(h.get("epoch"), h["loss"]) for h in history if "loss" in h and "eval_loss" not in h and h.get("epoch") is not None]
eval_pts  = [(h.get("epoch"), h["eval_loss"]) for h in history if "eval_loss" in h and h.get("epoch") is not None]

print(f"log_history: {len(train_pts)} training points, {len(eval_pts)} validation points")
if eval_pts:
    print(f"epoch range: {min(e for e, _ in eval_pts):.1f} -> {max(e for e, _ in eval_pts):.1f}")

fig, ax1 = plt.subplots(figsize=(10, 5))
if train_pts:
    ax1.plot([e for e, _ in train_pts], [l for _, l in train_pts],
             color="tab:blue", alpha=0.55, linewidth=1, label="Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()   # shares the SAME x-axis -- only the y-scale differs
if eval_pts:
    ax2.plot([e for e, _ in eval_pts], [l for _, l in eval_pts],
             color="tab:orange", marker="o", linewidth=2, label="Validation loss")
ax2.set_ylabel("Validation loss", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

# Mark where this session picked up from the checkpoint.
if epochs_done > 0:
    ax1.axvline(epochs_done, color="grey", linestyle="--", alpha=0.7)
    ax1.text(epochs_done, ax1.get_ylim()[1], "  resume", va="top", fontsize=9, color="grey")

lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc="upper right")
plt.title(f"{MODEL_KEY} — training curve, epochs 1–{len(eval_pts)}")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/{MODEL_KEY}_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPer-epoch validation loss (full history, epochs 1-N):")
prev = None
for i, (ep, vl) in enumerate(eval_pts, 1):
    delta = f"{vl - prev:+.4f}" if prev is not None else "    --"
    print(f"  Epoch {i} (epoch={ep:.2f}): {vl:.4f}   {delta}")
    prev = vl


## Step 8 — Save the fine-tuned model

`tie_weights()` here, after training, when the optimizer no longer matters — so the saved model
stores its embedding matrix once instead of four times.

In [ ]:
if TEST_MODE:
    print("TEST_MODE -- skipping model save (nothing was trained).")
    model_save_path = None
else:
    model_save_path = f"{RESULTS_DIR}/{MODEL_KEY}_finetuned"
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    size_gb = sum(os.path.getsize(os.path.join(model_save_path, f))
                  for f in os.listdir(model_save_path)) / 1024**3
    print(f"Saved to {model_save_path} ({size_gb:.2f} GB)")


## Step 9 — Scoring protocols

One definition, four protocols. `rouge_score`'s stock tokenizer uses `NON_ALPHANUM_RE =
[^a-z0-9]+`, which reduces Arabic to an empty token list and returns exactly 0.0; its
`tokenizer=` argument also needs an object with `.tokenize()`, not a lambda. Both failures are
silent, so the self-test below asserts each protocol is non-zero on known-matching text.

In [ ]:
_PUNCT_RE = re.compile(r"([^\w\s])", flags=re.UNICODE)


def _basic_tokenize(text):
    """Lowercase, whitespace-split, split punctuation off as separate tokens (XL-Sum's BasicTokenizer)."""
    text = text.lower()
    text = _PUNCT_RE.sub(r" \1 ", text)
    return [t for t in text.split() if t]


class SplitTokenizer:
    name = "whitespace"
    def tokenize(self, text):
        return text.split()


class PunctTokenizer:
    name = "punct-aware"
    def tokenize(self, text):
        return _basic_tokenize(text)


class StemmedTokenizer:
    """Matches Kahla et al. (2023) via XL-Sum's multilingual_rouge_scoring."""
    name = "punct-aware + stemmed"
    def __init__(self):
        self.stemmer = SnowballStemmer("arabic", ignore_stopwords=True)
    def tokenize(self, text):
        return [self.stemmer.stem(t) for t in _basic_tokenize(text)]


class LemmaTokenizer:
    """Lemmatisation resolves to real dictionary entries; the stemmer truncates
    (باريس -> اريس). ~180x slower per call, so results are memoised."""
    name = "punct-aware + lemmatized"
    def __init__(self):
        self._lem = qalsadi.lemmatizer.Lemmatizer()
        self._cache = {}
    def _lemma(self, w):
        if w not in self._cache:
            try:
                self._cache[w] = self._lem.lemmatize(w) or w
            except Exception:
                self._cache[w] = w
        return self._cache[w]
    def tokenize(self, text):
        return [self._lemma(t) for t in _basic_tokenize(text)]


PROTOCOLS = [SplitTokenizer(), PunctTokenizer(), StemmedTokenizer(), LemmaTokenizer()]
ROUGE_KEYS = ["rouge1", "rouge2", "rougeL", "rougeLsum"]


def rouge_with(predictions, references, tokenizer_obj):
    scorer = rouge_scorer.RougeScorer(ROUGE_KEYS, use_stemmer=False, tokenizer=tokenizer_obj)
    totals = {k: [] for k in ROUGE_KEYS}
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        for k in ROUGE_KEYS:
            totals[k].append(s[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in totals.items()}


def compute_bertscore(predictions, references, lang="ar"):
    bs = evaluate.load("bertscore")
    r = bs.compute(predictions=predictions, references=references, lang=lang)
    return r  # per-example lists, so slices can be averaged without recomputing


_p = "أعلن متحدث باسم فرق الإطفاء أنّ الحريق الضخم اندلع في كاتدرائية نوتردام"
_r = "أعلنت فرق الإطفاء أنّ الحريق الضخم الذي اندلع في كاتدرائية نوتردام في باريس"
print("Self-test (all must be > 0):")
for _t in PROTOCOLS:
    _v = rouge_with([_p], [_r], _t)["rouge1"]
    print(f"  {_t.name:<26} rouge1 = {_v:.4f}")
    assert _v > 0, f"{_t.name} returned zero on matching Arabic -- tokenizer broken"


## Step 10 — Evaluate on the full test set

Identical protocol for every model: same test examples, same generation settings, same four
scoring protocols. This is what makes the cross-model numbers comparable.

In [ ]:
test_articles = test_df["article"].tolist()
test_references = test_df["summary"].tolist()
source_vals = test_df["source"].tolist() if "source" in test_df.columns else ["all"] * len(test_df)
assert len(test_articles) == len(test_df), "evaluating on a subset -- use the full test set"


def generate_summaries(model, tokenizer, articles, batch_size=BATCH_SIZE_GEN):
    model.eval()
    preds = []
    for i in tqdm(range(0, len(articles), batch_size), desc="Generating"):
        batch = articles[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                           padding=True, max_length=MAX_INPUT_LEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_length=GEN_MAX_LEN,
                                 num_beams=NUM_BEAMS, early_stopping=True)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds


def arabic_ratio(text):
    """Fraction of characters in the Arabic Unicode block (U+0600-U+06FF)."""
    if not text or not text.strip():
        return 0.0
    return sum(1 for ch in text if "؀" <= ch <= "ۿ") / len(text)


# --- Sanity gate: 3 generations (~5s) before committing to 1000 (~8 min). ---
# An interrupted trainer.train() leaves the model emitting non-Arabic token spam
# ("ILLARILLARILLAR..."). Catching that here costs seconds instead of minutes.
_probe = generate_summaries(model, tokenizer, test_articles[:3])
_ratios = [arabic_ratio(p) for p in _probe]
print("Sanity probe -- Arabic character ratio per sample:")
for _i, (_p, _r) in enumerate(zip(_probe, _ratios)):
    print(f"  [{_i}] ratio={_r:.2f}  {_p[:90]}")

assert np.mean(_ratios) > 0.5, (
    "Model output is not Arabic (mean ratio "
    f"{np.mean(_ratios):.2f}). This is what a corrupted in-memory model looks like -- almost "
    "always because trainer.train() was interrupted. The checkpoint on disk is fine: restart "
    "the kernel and either re-run training uninterrupted, or set TEST_MODE = True."
)
print(f"Sanity probe passed (mean Arabic ratio {np.mean(_ratios):.2f}).")

if TEST_MODE:
    test_articles = test_articles[:TEST_MODE_N]
    test_references = test_references[:TEST_MODE_N]
    source_vals = source_vals[:TEST_MODE_N]
    print(f"TEST_MODE: evaluating on {len(test_articles)} examples only.")

t0 = time.time()
predictions = generate_summaries(model, tokenizer, test_articles)
generation_time_s = time.time() - t0

empty = sum(1 for p in predictions if not p.strip())
print(f"\nGenerated {len(predictions)} summaries in {generation_time_s:.0f}s | empty: {empty}")

print("\n=== Sample predictions (read these before trusting any metric) ===")
for i in range(3):
    print(f"\n[{i}] PRED: {predictions[i][:200]}")
    print(f"[{i}] REF : {test_references[i][:200]}")


In [ ]:
slices = {"ALL": list(range(len(predictions)))}
for src in sorted(set(source_vals)):
    slices[src] = [i for i, s in enumerate(source_vals) if s == src]

rows = []
for slice_name, idx in slices.items():
    sp = [predictions[i] for i in idx]
    sr = [test_references[i] for i in idx]
    for tok in PROTOCOLS:
        r = rouge_with(sp, sr, tok)
        rows.append({"slice": slice_name, "n": len(idx), "protocol": tok.name,
                     "ROUGE-1": r["rouge1"] * 100, "ROUGE-2": r["rouge2"] * 100,
                     "ROUGE-L": r["rougeL"] * 100})

results_df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")
print(results_df.to_string(index=False))


In [ ]:
METRIC_COLS = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
all_rows = results_df[results_df["slice"] == "ALL"].set_index("protocol")[METRIC_COLS]

print(f"{'protocol':<28}{'ROUGE-1':>10}{'ROUGE-2':>10}{'ROUGE-L':>10}")
for t in PROTOCOLS:
    r = all_rows.loc[t.name]
    print(f"{t.name:<28}{r['ROUGE-1']:>10.2f}{r['ROUGE-2']:>10.2f}{r['ROUGE-L']:>10.2f}")
print("-" * 58)
total = all_rows.loc["punct-aware + lemmatized"] - all_rows.loc["whitespace"]
lemma_vs_stem = all_rows.loc["punct-aware + lemmatized"] - all_rows.loc["punct-aware + stemmed"]
print(f"{'lemmatized - stemmed':<28}{lemma_vs_stem['ROUGE-1']:>+10.2f}{lemma_vs_stem['ROUGE-2']:>+10.2f}{lemma_vs_stem['ROUGE-L']:>+10.2f}")
print(f"{'TOTAL swing':<28}{total['ROUGE-1']:>+10.2f}{total['ROUGE-2']:>+10.2f}{total['ROUGE-L']:>+10.2f}")
rel = [((all_rows.loc["punct-aware + lemmatized"][m] / all_rows.loc["whitespace"][m] - 1) * 100
        if all_rows.loc["whitespace"][m] > 0 else float("nan")) for m in METRIC_COLS]
print(f"{'TOTAL relative':<28}{rel[0]:>+9.1f}%{rel[1]:>+9.1f}%{rel[2]:>+9.1f}%")


In [ ]:
bs_raw = compute_bertscore(predictions, test_references)
bert_rows = [{"slice": s, "n": len(idx),
              "precision": float(np.mean([bs_raw["precision"][i] for i in idx])),
              "recall": float(np.mean([bs_raw["recall"][i] for i in idx])),
              "f1": float(np.mean([bs_raw["f1"][i] for i in idx]))}
             for s, idx in slices.items()]
bert_df = pd.DataFrame(bert_rows)
print(bert_df.to_string(index=False))


## Step 11 — Save results

In [ ]:
# TEST_MODE writes to a separate prefix so a smoke run can never overwrite real results.
SUFFIX = "_TESTMODE" if TEST_MODE else ""

pd.DataFrame({
    "article": test_articles,
    "reference": test_references,
    "prediction": predictions,
    "source": source_vals,
}).to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_predictions.csv", index=False)

results_df.to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_rouge_by_protocol.csv", index=False)
bert_df.to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_bertscore.csv", index=False)

payload = {
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "run_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data_manifest_created_at": data_manifest["created_at_utc"],
    "resumed_from": RESUME_FROM_CHECKPOINT,
    "epochs_in_checkpoint": epochs_done,
    "hyperparameters": {
        "learning_rate": LEARNING_RATE, "num_epochs": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "max_input_len": MAX_INPUT_LEN, "max_target_len": MAX_TARGET_LEN,
        "num_beams": NUM_BEAMS, "seed": RANDOM_SEED, "fp16": False,
        "optimizer": "Adafactor(scale_parameter=False, relative_step=False, warmup_init=False)",
    },
    "dataset_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "training": {
        "test_mode": TEST_MODE,
        "training_completed": TRAINING_COMPLETED,
        "final_train_loss": (train_result.training_loss if train_result is not None else None),
        "log_history": history,
        "validation_loss_per_epoch": [l for _, l in eval_pts],
    },
    "rouge_by_protocol": results_df.to_dict(orient="records"),
    "bertscore_by_slice": bert_df.to_dict(orient="records"),
    "generation": {"time_s": generation_time_s, "empty_outputs": int(empty)},
    "evaluation_notes": (
        "ROUGE via rouge_score with Unicode-safe tokenizers; the stock default empties Arabic. "
        "'stemmed' replicates Kahla et al. (2023) / XL-Sum multilingual_rouge_scoring. All "
        "protocols share one tokenization base, so normalisation is the only variable."
    ),
}

with open(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_results.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Saved to {RESULTS_DIR}/:")
for fn in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {fn}")


## Next model

Change `MODEL_KEY` in Step 0, restart the kernel, run again. Outputs are namespaced by
`MODEL_KEY`, so runs never overwrite each other.

Suggested order: **AraBART first** — it uses fp16 and is the smallest of the three, so it
finishes fastest and validates the notebook end-to-end before you commit to AraT5, which runs at
fp32 and will probably need more than one 12-hour session (use `RESUME_FROM_CHECKPOINT`).

Jais-6.7B stays in its own notebook: QLoRA changes the model loading, the tokenisation (causal
LM with prompt masking), and the saving (adapter only), so folding it in here would add branching
that obscures the shared path.